# BERTScore (semantic evaluation) for baseline vs BRIO checkpoints

BERTScore compares the meaning of a summary with the reference summary using BERT embeddings, so it complements ROUGE (word overlap).

- **Part A** runs on the 100 articles used for the Gemini judge. Quick.
- **Part B** generates summaries for the whole test set (10,972 articles) and scores them. This is the version to report in the paper.

Several BRIO checkpoints can be evaluated in one run: list them in `BRIO_CKPTS` in the Paths cell. They are processed one after another, and each one is compared with the baseline.

In [1]:
# Install condacolab. The runtime restarts after this cell, that is normal.
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:12
🔁 Restarting kernel...


### Env

In [1]:
# Clone the repo and mount Google Drive.
!git clone https://github.com/alvrian/BRIO.git

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%cd BRIO

Cloning into 'BRIO'...
remote: Enumerating objects: 471, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 471 (delta 132), reused 98 (delta 71), pack-reused 291 (from 2)
Receiving objects: 100% (471/471), 15.99 MiB | 20.58 MiB/s, done.
Resolving deltas: 100% (286/286), done.
Mounted at /content/drive
/content/BRIO


In [2]:
# Same env setup as BRIO_1_1.ipynb, plus the bert-score package and the fine-tuned IndoBART baseline.
! conda create --name env --file spec-file.txt
! conda run --no-capture-output -n env pip install -r requirements.txt
! conda run -n env python -c "import nltk;nltk.download('punkt')"
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run --no-capture-output -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install
! conda run --no-capture-output -n env pip install bert-score
print("done setting up env")
! cp /content/drive/MyDrive/BRIO/indobart-liputan6-finetuned.zip .
! unzip -q indobart-liputan6-finetuned.zip
! rm indobart-liputan6-finetuned.zip
print("all done")

Retrieving notices: - done

_libgcc_mutex-0.1    | :   0% 0/1 [00:00<?, ?it/s]
blas-1.0             | :   0% 0/1 [00:00<?, ?it/s]

ca-certificates-2021 | :   0% 0/1 [00:00<?, ?it/s]


intel-openmp-2021.3. | :   0% 0/1 [00:00<?, ?it/s]



ld_impl_linux-64-2.3 | :   0% 0/1 [00:00<?, ?it/s]




libstdcxx-ng-9.3.0   | :   0% 0/1 [00:00<?, ?it/s]





libgomp-9.3.0        | :   0% 0/1 [00:00<?, ?it/s]






mkl-2021.3.0         | :   0% 0/1 [00:00<?, ?it/s]







_openmp_mutex-4.5    | :   0% 0/1 [00:00<?, ?it/s]








libgcc-ng-9.3.0      | :   0% 0/1 [00:00<?, ?it/s]









bzip2-1.0.8          | :   0% 0/1 [00:00<?, ?it/s]










cudatoolkit-11.1.1   | :   0% 0/1 [00:00<?, ?it/s]











giflib-5.2.1         | :   0% 0/1 [00:00<?, ?it/s]












gmp-6.2.1            | :   0% 0/1 [00:00<?, ?it/s]













jpeg-9b              | :   0% 0/1 [00:00<?, ?it/s]














lame-3.100           | :   0% 0/1 [00:00<?, ?it/s]















libffi-3.3           | :   0% 0/1 

### Paths

In [ ]:
# List every model_generation.bin saved on Drive, to copy the right paths into the next cell.
import glob
for p in sorted(glob.glob("/content/drive/MyDrive/BRIO/checkpoints/*/*/model_generation.bin")):
    print(p)

/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1/4000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/10000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/12000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/2000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/4000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/6000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/8000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/master_1_0/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_third_seed/10000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/B

In [3]:
import os

# Liputan6 canonical test set on Drive.
TEST_SOURCE = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source"
TEST_TARGET = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target"

# Fine-tuned IndoBART baseline, unzipped by the env cell.
BASELINE = "./indobart-liputan6-finetuned"

# BRIO checkpoints to evaluate: short name -> path of model_generation.bin.
# EDIT: add or remove lines. Keep the name "brio" for the checkpoint you used with the Gemini judge,
# so its saved summaries are reused.
CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints"
# BRIO_CKPTS = {
#     "brio": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin",
#     "brio_seed3": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin",
#     "brio_cand8": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_8/master_1_1/model_generation.bin",
#     "brio_cand12": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_12/master_1_1/model_generation.bin",
# }

BRIO_CKPTS = {
    "brio_1_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin",
    "brio_1_1_seed2":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin",
    "brio_1_1_seed3":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin",
    "brio_10_1_seed1":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1-reduceLR/8000/model_generation.bin",
    "brio_10_1_seed2":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_second_seed/master_1_10/model_generation.bin",
    "brio_10_1_seed3":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_third_seed/master_1_10/model_generation.bin",
    "brio_100_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-reduceLR/4000/model_generation.bin",
    "brio_100_1_seed2": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-second_seed/master_1_100/model_generation.bin",
    "brio_100_1_seed3": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-third_seed/master_1_100/model_generation.bin",
}

# Folder with the 100-article sample (Part A) and a folder for the whole test set (Part B).
SAMPLE_DIR = "/content/drive/MyDrive/BRIO/eval_outputs"
FULL_DIR = "/content/drive/MyDrive/BRIO/eval_outputs_full"

for p in [TEST_SOURCE, TEST_TARGET, BASELINE] + list(BRIO_CKPTS.values()):
    print("found  " if os.path.exists(p) else "MISSING", p)

# Build the system list and the baseline-vs-checkpoint pairs once, used by every scoring cell.
SYSTEMS = " ".join(["baseline"] + list(BRIO_CKPTS))
PAIRS = " ".join("--pair baseline " + name for name in BRIO_CKPTS)
print(SYSTEMS)

found   /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source
found   /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target
found   ./indobart-liputan6-finetuned
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1-reduceLR/8000/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1_second_seed/master_1_10/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1_third_seed/master_1_10/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_100-1-reduceLR/4000/model_gener

In [4]:
# Same helper as before, with the batch size as a parameter.
def generate_all(out_dir, sample_n, batch_size=8):
    models = {"baseline": BASELINE}
    models.update(BRIO_CKPTS)
    for name, path in models.items():
        print("=====", name)
        !conda run --no-capture-output -n env python generate_outputs.py --name {name} --model_path "{path}" --src_file "{TEST_SOURCE}" --tgt_file "{TEST_TARGET}" --out_dir "{out_dir}" --sample_n {sample_n} --batch_size {batch_size}


### Part A: the 100 sampled articles

In [ ]:
# Summaries for the 100 sampled articles (only the checkpoints not saved yet are generated).
generate_all(SAMPLE_DIR, 1)

In [ ]:
# BERTScore for every system on the 100 articles. Results are saved in SAMPLE_DIR/bertscore/.
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir "{SAMPLE_DIR}" --systems {SYSTEMS} {PAIRS}

### Part B: whole test set

Generation is the slow part and the time grows with every checkpoint in the list. If Colab disconnects, run the same cell again: finished models are skipped and the interrupted one continues from its last saved summary.

In [ ]:
# Summaries for all 10,972 test articles, for the baseline and every checkpoint.
# generate_all(FULL_DIR, 0)
generate_all(FULL_DIR, 0, batch_size=16)

===== baseline
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/baseline.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to /content/drive/MyDrive/BRIO/eval_outputs_full/baseline.out
ROUGE-1 0.3937 | ROUGE-2 0.2165 | ROUGE-L 0.3649  (n=10972)
===== brio_1_1_seed1
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed1.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to /content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed1.out
ROUGE-1 0.4119 | ROUGE-2 0.2274 | ROUGE-L 0.3814  (n=10972)
===== brio_1_1_seed2
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed2.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to 

In [ ]:
# BERTScore for every system on the whole test set. Results are saved in FULL_DIR/bertscore/.
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir "{FULL_DIR}" --systems {SYSTEMS} {PAIRS}

In [ ]:
# Show the result table (ROUGE and BERTScore side by side, one row per system) and the per-article scores.
import pandas as pd
print(open(os.path.join(FULL_DIR, "bertscore", "bertscore_summary.txt"), encoding="utf-8").read())
scores = pd.read_csv(os.path.join(FULL_DIR, "bertscore", "bertscore.csv"))
scores.head()

### Optional: IndoBERT instead of multilingual BERT

The default embedding model is multilingual BERT. To check that the result does not depend on that choice, run the same scoring with IndoBERT. This overwrites the files in `bertscore/`, so copy `bertscore_summary.txt` first if you want to keep both.

In [ ]:
# Same scoring with IndoBERT embeddings (layer 9, the layer bert-score uses for BERT-base models).
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir "{FULL_DIR}" --systems {SYSTEMS} {PAIRS} --model_type indobenchmark/indobert-base-p1 --num_layers 9